# Machine Learning Champion Models (XGBoost Framework)

Gradient-boosted decision tree models: Poisson Frequency with base_margin offset, Gamma Severity, Two-Part Pure Premium, and Direct Tweedie XGBoost.

> Run notebooks in order 01 → 06: later notebooks load artifacts saved by earlier ones.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

import xgboost as xgb


FEATURE_COLS = [
    "DrivAge",
    "VehAge",
    "VehPower",
    "BonusMalus",
    "LogDensity",
    "Area",
    "VehGas",
    "VehBrand",
    "Region",
]

CATEGORICAL_COLS = ["Area", "VehGas", "VehBrand", "Region"]


def prepare_xgb_dataframe(df: pd.DataFrame, feature_cols: list[str] = None) -> pd.DataFrame:
    if feature_cols is None:
        feature_cols = FEATURE_COLS
    X = df[feature_cols].copy()
    for col in CATEGORICAL_COLS:
        if col in X.columns:
            X[col] = X[col].astype("category")
    return X


# Alias for backwards compatibility
prepare_xgb_features = prepare_xgb_dataframe


class FrequencyPoissonXGB:

    def __init__(
        self,
        max_depth: int = 4,
        learning_rate: float = 0.05,
        n_estimators: int = 300,
        subsample: float = 0.8,
        colsample_bytree: float = 0.8,
        min_child_weight: float = 20.0,
        random_state: int = 42,
        enforce_monotonic_bm: bool = True,
        target_col: str = "AttritionalClaimNb_Capped",
    ):
        self.feature_cols = FEATURE_COLS
        self.n_estimators = n_estimators
        self.target_col = target_col
        
        monotone_tuple = [0] * len(self.feature_cols)
        if enforce_monotonic_bm and "BonusMalus" in self.feature_cols:
            idx = self.feature_cols.index("BonusMalus")
            monotone_tuple[idx] = 1

        self.params = {
            "objective": "count:poisson",
            "eval_metric": "poisson-nloglik",
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "seed": random_state,
            "tree_method": "hist",
            "monotone_constraints": tuple(monotone_tuple),
        }
        self.booster = None

    def fit(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame = None,
        early_stopping_rounds: int = 25,
    ):
        X_tr = prepare_xgb_dataframe(df_train, self.feature_cols)
        label_col = self.target_col if self.target_col in df_train.columns else "ClaimNb"
        dtrain = xgb.DMatrix(
            X_tr,
            label=df_train[label_col].values,
            base_margin=df_train["LogExposure"].values,
            enable_categorical=True,
        )

        evals = [(dtrain, "train")]
        if df_val is not None:
            X_val = prepare_xgb_dataframe(df_val, self.feature_cols)
            label_val = self.target_col if self.target_col in df_val.columns else "ClaimNb"
            dval = xgb.DMatrix(
                X_val,
                label=df_val[label_val].values,
                base_margin=df_val["LogExposure"].values,
                enable_categorical=True,
            )
            evals.append((dval, "val"))
            self.booster = xgb.train(
                self.params,
                dtrain,
                num_boost_round=self.n_estimators,
                evals=evals,
                early_stopping_rounds=early_stopping_rounds,
                verbose_eval=False,
            )
        else:
            self.booster = xgb.train(
                self.params,
                dtrain,
                num_boost_round=self.n_estimators,
                evals=evals,
                verbose_eval=False,
            )
        return self

    def predict(self, df: pd.DataFrame, offset_col: str = None) -> np.ndarray:
        X = prepare_xgb_dataframe(df, self.feature_cols)
        if offset_col is not None:
            base_margin = df[offset_col].values
        else:
            # Annualized frequency (exposure = 1.0, offset = 0.0)
            base_margin = np.zeros(len(df))
            
        dmat = xgb.DMatrix(X, base_margin=base_margin, enable_categorical=True)
        return self.booster.predict(dmat)


class SeverityGammaXGB:

    def __init__(
        self,
        max_depth: int = 3,
        learning_rate: float = 0.03,
        n_estimators: int = 150,
        subsample: float = 0.8,
        colsample_bytree: float = 0.8,
        min_child_weight: float = 15.0,
        random_state: int = 42,
        large_loss_threshold: float = 100_000.0,
    ):
        self.feature_cols = FEATURE_COLS
        self.n_estimators = n_estimators
        self.large_loss_threshold = large_loss_threshold
        self.params = {
            "objective": "reg:gamma",
            "eval_metric": "gamma-deviance",
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "seed": random_state,
            "tree_method": "hist",
        }
        self.booster = None

    def _filter_claims(self, df: pd.DataFrame) -> pd.DataFrame:
        return df[
            (df["HasAttritionalLoss"] == 1) &
            (df["AvgClaimSeverity"] > 0) &
            (df["AvgClaimSeverity"] < self.large_loss_threshold) &
            (df["AttritionalClaimAmount"] > 0)
        ].copy()

    def fit(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame = None,
        early_stopping_rounds: int = 20,
    ):
        claims_train = self._filter_claims(df_train)

        X_tr = prepare_xgb_dataframe(claims_train, self.feature_cols)
        weights_tr = claims_train["AttritionalSevRecordCount"].clip(lower=1).values
        dtrain = xgb.DMatrix(
            X_tr,
            label=claims_train["AvgClaimSeverity"].values,
            weight=weights_tr,
            enable_categorical=True,
        )

        evals = [(dtrain, "train")]
        if df_val is not None:
            claims_val = self._filter_claims(df_val)
            if len(claims_val) > 0:
                X_val = prepare_xgb_dataframe(claims_val, self.feature_cols)
                weights_val = claims_val["SevRecordCount"].clip(lower=1).values
                dval = xgb.DMatrix(
                    X_val,
                    label=claims_val["AvgClaimSeverity"].values,
                    weight=weights_val,
                    enable_categorical=True,
                )
                evals.append((dval, "val"))
                self.booster = xgb.train(
                    self.params,
                    dtrain,
                    num_boost_round=self.n_estimators,
                    evals=evals,
                    early_stopping_rounds=early_stopping_rounds,
                    verbose_eval=False,
                )
                return self

        # No validation set — train for fixed rounds
        self.booster = xgb.train(
            self.params,
            dtrain,
            num_boost_round=self.n_estimators,
            evals=evals,
            verbose_eval=False,
        )
        return self

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        X = prepare_xgb_dataframe(df, self.feature_cols)
        dmat = xgb.DMatrix(X, enable_categorical=True)
        return self.booster.predict(dmat)


class TweedieXGB:

    def __init__(
        self,
        tweedie_variance_power: float = 1.5,
        max_depth: int = 4,
        learning_rate: float = 0.05,
        n_estimators: int = 250,
        subsample: float = 0.8,
        colsample_bytree: float = 0.8,
        min_child_weight: float = 20.0,
        random_state: int = 42,
    ):
        self.feature_cols = FEATURE_COLS
        self.n_estimators = n_estimators
        self.params = {
            "objective": "reg:tweedie",
            "tweedie_variance_power": tweedie_variance_power,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "seed": random_state,
            "tree_method": "hist",
        }
        self.booster = None

    def fit(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame = None,
        early_stopping_rounds: int = 25,
    ):
        X_tr = prepare_xgb_dataframe(df_train, self.feature_cols)
        weights_tr = df_train["Exposure_Capped"].values
        dtrain = xgb.DMatrix(
            X_tr,
            label=df_train["PurePremium"].values,
            weight=weights_tr,
            enable_categorical=True,
        )

        evals = [(dtrain, "train")]
        if df_val is not None:
            X_val = prepare_xgb_dataframe(df_val, self.feature_cols)
            weights_val = df_val["Exposure_Capped"].values
            dval = xgb.DMatrix(
                X_val,
                label=df_val["PurePremium"].values,
                weight=weights_val,
                enable_categorical=True,
            )
            evals.append((dval, "val"))
            self.booster = xgb.train(
                self.params,
                dtrain,
                num_boost_round=self.n_estimators,
                evals=evals,
                early_stopping_rounds=early_stopping_rounds,
                verbose_eval=False,
            )
        else:
            self.booster = xgb.train(
                self.params,
                dtrain,
                num_boost_round=self.n_estimators,
                evals=evals,
                verbose_eval=False,
            )
        return self

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        X = prepare_xgb_dataframe(df, self.feature_cols)
        dmat = xgb.DMatrix(X, enable_categorical=True)
        return self.booster.predict(dmat)


class TwoPartXGB:

    def __init__(self, freq_xgb: FrequencyPoissonXGB = None, sev_xgb: SeverityGammaXGB = None):
        self.freq_xgb = freq_xgb if freq_xgb is not None else FrequencyPoissonXGB()
        self.sev_xgb = sev_xgb if sev_xgb is not None else SeverityGammaXGB()

    def fit(self, df_train: pd.DataFrame, df_val: pd.DataFrame = None):
        self.freq_xgb.fit(df_train, df_val=df_val)
        self.sev_xgb.fit(df_train, df_val=df_val)
        return self

    def predict_pure_premium(self, df: pd.DataFrame) -> np.ndarray:
        pred_freq = self.freq_xgb.predict(df, offset_col=None)
        pred_sev = self.sev_xgb.predict(df)
        return pred_freq * pred_sev


def poisson_deviance(y_true, y_pred, exposure=None) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 1e-9, None)
    
    # Safely evaluate log only where y_true > 0
    safe_y_true = np.where(y_true > 0, y_true, 1.0)
    dev = np.where(
        y_true > 0,
        2 * (y_true * np.log(safe_y_true / y_pred) - (y_true - y_pred)),
        2 * y_pred
    )
    if exposure is not None:
        exposure = np.asarray(exposure, dtype=float)
        return float(np.sum(exposure * dev) / np.sum(exposure))
    return float(np.mean(dev))


def gamma_deviance(y_true, y_pred, weights=None) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 1e-9, None)
    safe_y_true = np.clip(y_true, 1e-9, None)
    
    dev = 2 * ((y_true - y_pred) / y_pred - np.log(safe_y_true / y_pred))
    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        return float(np.sum(weights * dev) / np.sum(weights))
    return float(np.mean(dev))


def tweedie_deviance(y_true, y_pred, p: float = 1.5, weights=None) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 1e-9, None)
    
    if p == 1.0:
        safe_y_true = np.where(y_true > 0, y_true, 1.0)
        dev = np.where(
            y_true > 0,
            2 * (y_true * np.log(safe_y_true / y_pred) - (y_true - y_pred)),
            2 * y_pred
        )
    elif p == 2.0:
        safe_y_true = np.clip(y_true, 1e-9, None)
        dev = 2 * (np.log(y_pred / safe_y_true) + y_true / y_pred - 1)
    else:
        term1 = (np.power(np.clip(y_true, 0, None), 2 - p)) / ((1 - p) * (2 - p))
        term2 = (y_true * np.power(y_pred, 1 - p)) / (1 - p)
        term3 = (np.power(y_pred, 2 - p)) / (2 - p)
        dev = 2 * (term1 - term2 + term3)
        
    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        return float(np.sum(weights * dev) / np.sum(weights))
    return float(np.mean(dev))


def gini_coefficient(y_actual, y_pred, exposure=None) -> float:
    y_actual = np.asarray(y_actual, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(y_actual)
    
    if exposure is None:
        weights = np.ones(n, dtype=float)
    else:
        weights = np.asarray(exposure, dtype=float)

    # No ranking possible when all predictions are identical
    if np.all(y_pred == y_pred[0]):
        return 0.0
         
    order = np.argsort(y_pred, kind="mergesort")
    sorted_actual = y_actual[order]
    sorted_weights = weights[order]
    
    cum_actual = np.cumsum(sorted_actual * sorted_weights)
    total_actual = cum_actual[-1]
    if total_actual == 0:
        return 0.0
    # Include (0,0) origin so area is exact even for small n
    cum_actual_norm = np.insert(cum_actual / total_actual, 0, 0.0)
    
    cum_weights = np.cumsum(sorted_weights)
    cum_weights_norm = np.insert(cum_weights / cum_weights[-1], 0, 0.0)
    
    lorenz_area = np.sum((cum_actual_norm[:-1] + cum_actual_norm[1:]) * np.diff(cum_weights_norm)) / 2.0
    gini = 1.0 - 2.0 * lorenz_area
    return float(gini)

def train_val_test_split(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df_trainval, df_test = train_test_split(
        df, test_size=test_size, random_state=random_state,
        stratify=df["HasAttritionalLoss"],
    )
    val_fraction = val_size / (1 - test_size)
    df_train, df_val = train_test_split(
        df_trainval, test_size=val_fraction, random_state=random_state,
        stratify=df_trainval["HasAttritionalLoss"],
    )
    logger.info("Split sizes — Train: %d, Val: %d, Test: %d", len(df_train), len(df_val), len(df_test))
    return df_train, df_val, df_test


# ---------- tiny config readers (replace the old factory) ----------
def split_abt(df, config):
    s = config.get("split", {})
    return train_val_test_split(df, test_size=s.get("test_size", 0.2),
                                val_size=s.get("val_size", 0.1),
                                random_state=config.get("project", {}).get("random_seed", 42))

def get_band(config, key):
    spec = config.get("bands", {}).get(key, {})
    return spec.get("bins", []), spec.get("labels", [])

def build_frequency_xgb(config):
    m = dict(config["models"]["xgb_frequency"])
    es = m.pop("early_stopping_rounds", 25)
    m.setdefault("random_state", config.get("project", {}).get("random_seed", 42))
    return FrequencyPoissonXGB(**m), es

def build_severity_xgb(config):
    m = dict(config["models"]["xgb_severity"])
    es = m.pop("early_stopping_rounds", 20)
    m.pop("large_loss_threshold", None)
    m.setdefault("random_state", config.get("project", {}).get("random_seed", 42))
    m["large_loss_threshold"] = config.get("cleaning", {}).get("large_loss_threshold", 100000.0)
    return SeverityGammaXGB(**m), es

def build_tweedie_xgb(config):
    m = dict(config["models"]["xgb_tweedie"])
    es = m.pop("early_stopping_rounds", 25)
    m.setdefault("random_state", config.get("project", {}).get("random_seed", 42))
    return TweedieXGB(**m), es
import yaml
import logging
logger = logging.getLogger(__name__)
from sklearn.model_selection import train_test_split

def load_config(path="config/config.yaml"):
    with open(path) as f:
        return yaml.safe_load(f)

## 1. Train / Validation / Test Split Protocol (70 / 10 / 20)

In [2]:
config = load_config("../config/config.yaml")
seed = config["project"]["random_seed"]
LL = config["cleaning"]["large_loss_threshold"]
P = config["evaluation"]["tweedie_p"]
mpaths = {k: ("../" + v if isinstance(v, str) else v)
          for k, v in config["data"]["paths"].items()}  # config paths are project-relative; notebooks run from notebooks/
df = pd.read_parquet("../data/processed/abt_motor_portfolio.parquet")
df_train, df_val, test_df = split_abt(df, config)
train_df, val_df = df_train, df_val

# GLM baseline predictions come from notebook 03 — run it first if this file is missing
glm_pred_path = Path(mpaths["pred_glm_file"])
if not glm_pred_path.exists():
    raise RuntimeError(f"Missing {glm_pred_path}. Run notebook 03 first: it trains the GLMs and saves their predictions.")
test_df = test_df.merge(pd.read_parquet(glm_pred_path), on="IDpol", how="left")

print(f"Train set (70%):      {len(train_df):,} policies ({train_df['Exposure_Capped'].sum():,.1f} years)")
print(f"Validation set (10%): {len(val_df):,} policies ({val_df['Exposure_Capped'].sum():,.1f} years)")
print(f"Test set (20%):       {len(test_df):,} policies ({test_df['Exposure_Capped'].sum():,.1f} years)")

Train set (70%):      474,608 policies (250,937.9 years)
Validation set (10%): 67,802 policies (35,567.5 years)
Test set (20%):       135,603 policies (71,854.7 years)


## 2. Frequency Modeling: Poisson XGBoost vs Poisson GLM

In [3]:
# Fit Poisson XGBoost with early stopping on the validation set
xgb_freq, ES_FREQ = build_frequency_xgb(config)
xgb_freq.fit(train_df, df_val=val_df, early_stopping_rounds=ES_FREQ)

# Evaluate on Test Set (GLM predictions loaded from notebook 03)
test_xgb_freq_pred = xgb_freq.predict(test_df, offset_col="LogExposure")
xgb_freq_dev = poisson_deviance(test_df["AttritionalClaimNb_Capped"], test_xgb_freq_pred, exposure=test_df["Exposure_Capped"])
xgb_freq_gini = gini_coefficient(test_df["AttritionalClaimNb_Capped"], test_xgb_freq_pred, exposure=test_df["Exposure_Capped"])

test_glm_freq_pred = test_df["Pred_GLM_Freq"].values
glm_freq_dev = poisson_deviance(test_df["AttritionalClaimNb_Capped"], test_glm_freq_pred, exposure=test_df["Exposure_Capped"])
glm_freq_gini = gini_coefficient(test_df["AttritionalClaimNb_Capped"], test_glm_freq_pred, exposure=test_df["Exposure_Capped"])

freq_comparison = pd.DataFrame({
    "Model": ["Poisson GLM (Baseline)", "Poisson XGBoost (Champion)"],
    "Test Poisson Deviance": [glm_freq_dev, xgb_freq_dev],
    "Test Frequency Gini": [glm_freq_gini, xgb_freq_gini],
    "Deviance Reduction (%)": ["-", f"{(glm_freq_dev - xgb_freq_dev) / glm_freq_dev * 100:.2f}%"]
})
display(freq_comparison)

,Model,Test Poisson Deviance,Test Frequency Gini,Deviance Reduction (%)
0,Poisson GLM (Baseline),0.296338,0.288950,-
1,Poisson XGBoost (Champion),0.291189,0.322983,1.74%


## 3. Severity Modeling: Gamma XGBoost vs Gamma GLM

In [4]:
# Fit Gamma XGBoost with early stopping on the validation set
xgb_sev, ES_SEV = build_severity_xgb(config)
xgb_sev.fit(train_df, df_val=val_df, early_stopping_rounds=ES_SEV)

# Evaluate on test positive attritional claims (GLM predictions loaded from notebook 03)
test_claims = test_df[
    (test_df["HasAttritionalLoss"] == 1) &
    (test_df["AvgClaimSeverity"] > 0) &
    (test_df["AvgClaimSeverity"] < LL) &
    (test_df["AttritionalClaimAmount"] > 0)
].copy()

pred_xgb_sev = xgb_sev.predict(test_claims)
pred_glm_sev = test_claims["Pred_GLM_Sev"].values

xgb_sev_dev = gamma_deviance(test_claims["AvgClaimSeverity"], pred_xgb_sev, weights=test_claims["AttritionalSevRecordCount"])
glm_sev_dev = gamma_deviance(test_claims["AvgClaimSeverity"], pred_glm_sev, weights=test_claims["AttritionalSevRecordCount"])

sev_comparison = pd.DataFrame({
    "Model": ["Gamma GLM (Baseline)", "Gamma XGBoost (Champion)"],
    "Test Gamma Deviance": [glm_sev_dev, xgb_sev_dev],
    "Mean Predicted Severity (€)": [f"€{pred_glm_sev.mean():.2f}", f"€{pred_xgb_sev.mean():.2f}"]
})
display(sev_comparison)

,Model,Test Gamma Deviance,Mean Predicted Severity (€)
0,Gamma GLM (Baseline),1.096718,€1724.75
1,Gamma XGBoost (Champion),1.096283,€1696.99


## 4. Pure Premium Compound Models: Two-Part vs Direct Tweedie

In [5]:
# 1. Two-Part Models (GLM pure premium loaded from notebook 03)
pred_glm_pure = test_df["Pred_GLM_Pure"].values
pred_glm_pure_bal = test_df["Pred_GLM_Pure_Bal"].values
twopart_xgb = TwoPartXGB(xgb_freq, xgb_sev)
pred_xgb_pure = twopart_xgb.predict_pure_premium(test_df)

# 2. Direct Tweedie XGBoost with early stopping on the validation set
tweedie_xgb, ES_TW = build_tweedie_xgb(config)
tweedie_xgb.fit(train_df, df_val=val_df, early_stopping_rounds=ES_TW)
pred_tweedie_xgb_pure = tweedie_xgb.predict(test_df)

# Balance correction factors from TRAIN for the XGB pure premiums (deviance/Gini stay on raw predictions)
_train_loss = float(train_df["AttritionalClaimAmount"].sum())
_train_xgb = twopart_xgb.predict_pure_premium(train_df)
_train_twx = tweedie_xgb.predict(train_df)
bal_xgb = _train_loss / float((_train_xgb * train_df["Exposure_Capped"]).sum())
bal_twx = _train_loss / float((_train_twx * train_df["Exposure_Capped"]).sum())
pred_xgb_pure_bal = pred_xgb_pure * bal_xgb
pred_tweedie_xgb_pure_bal = pred_tweedie_xgb_pure * bal_twx
print(f"Balance factors (train): xgb two-part={bal_xgb:.4f}, tweedie={bal_twx:.4f}")

# Evaluation Metrics on Test Set (exposure-weighted means: fair comparison with observed €/year)
exp_w = test_df["Exposure_Capped"]
dev_glm_twopart = tweedie_deviance(test_df["PurePremium"], pred_glm_pure, p=P, weights=exp_w)
dev_xgb_twopart = tweedie_deviance(test_df["PurePremium"], pred_xgb_pure, p=P, weights=exp_w)
dev_xgb_tweedie = tweedie_deviance(test_df["PurePremium"], pred_tweedie_xgb_pure, p=P, weights=exp_w)

gini_glm_twopart = gini_coefficient(test_df["PurePremium"], pred_glm_pure, exposure=exp_w)
gini_xgb_twopart = gini_coefficient(test_df["PurePremium"], pred_xgb_pure, exposure=exp_w)
gini_xgb_tweedie = gini_coefficient(test_df["PurePremium"], pred_tweedie_xgb_pure, exposure=exp_w)

overall_comparison = pd.DataFrame({
    "Pure Premium Model Architecture": [
        "Two-Part GLM (Poisson x Gamma)",
        "Two-Part XGBoost (Poisson x Gamma)",
        f"Direct Tweedie XGBoost (p={P})"
    ],
    f"Test Tweedie Deviance (p={P})": [dev_glm_twopart, dev_xgb_twopart, dev_xgb_tweedie],
    "Test Gini Index": [gini_glm_twopart, gini_xgb_twopart, gini_xgb_tweedie],
    "Balanced Mean Predicted Attritional Pure Premium": [
        f"€{np.average(pred_glm_pure_bal, weights=exp_w):.2f}",
        f"€{np.average(pred_xgb_pure_bal, weights=exp_w):.2f}",
        f"€{np.average(pred_tweedie_xgb_pure_bal, weights=exp_w):.2f}"
    ]
})
display(overall_comparison)

# Save native boosters + test predictions for notebooks 05-06 (each model trained exactly once)
Path(mpaths["models_dir"]).mkdir(parents=True, exist_ok=True)
xgb_freq.booster.save_model(mpaths["xgb_freq_booster"])
xgb_sev.booster.save_model(mpaths["xgb_sev_booster"])
tweedie_xgb.booster.save_model(mpaths["xgb_tweedie_booster"])
test_xgb_freq = xgb_freq.predict(test_df, offset_col="LogExposure")
pd.DataFrame({
    "IDpol": test_df["IDpol"].values,
    "Pred_XGB_Freq": test_xgb_freq,
    "Pred_XGB_Sev": xgb_sev.predict(test_df),
    "Pred_XGB_Pure": pred_xgb_pure,
    "Pred_XGB_Pure_Bal": pred_xgb_pure_bal,
    "Pred_Tweedie_Pure_Bal": pred_tweedie_xgb_pure_bal,
    "Pred_Tweedie_Pure": pred_tweedie_xgb_pure,
}).to_parquet(mpaths["pred_xgb_file"], index=False)
import json as _json
_bal_path = mpaths["balance_factors_file"]
_bal_all = _json.load(open(_bal_path)) if Path(_bal_path).exists() else {}
_bal_all.update({"xgb_twopart": bal_xgb, "xgb_tweedie": bal_twx})
_json.dump(_bal_all, open(_bal_path, "w"), indent=1)
print(f"Saved XGB boosters + predictions to {mpaths['models_dir']}/ and {mpaths['pred_xgb_file']}")

Balance factors (train): xgb two-part=1.0100, tweedie=1.0839


,Pure Premium Model Architecture,Test Tweedie Deviance (p=1.5),Test Gini Index,Balanced Mean Predicted Attritional Pure Premium
0,Two-Part GLM (Poisson x Gamma),69.536118,0.313561,€127.08
1,Two-Part XGBoost (Poisson x Gamma),68.775705,0.339558,€127.05
2,Direct Tweedie XGBoost (p=1.5),68.714196,0.337946,€127.05


Saved XGB boosters + predictions to ../models/ and ../data/processed/pred_xgb.parquet
